<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB07_Machine_Learning_Fundamentals_First_Real_Ship_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB07 · Class 7 — Machine Learning Fundamentals with a Real Ship Dataset**

## Block 2: AI — Machine Learning (continued)

This class introduces the core vocabulary and workflow of Machine Learning (supervised learning, training/validation/test sets, overfitting, evaluation metrics) and then applies all of it, end to end, to a **real, publicly published dataset**: the [*Ship Fuel Consumption and CO2 Emissions Analysis*](https://www.kaggle.com/datasets/jeleeladekunlefijabi/ship-fuel-consumption-and-co2-emissions-analysis) dataset from Kaggle, covering 120 vessels operating on Nigerian waterways over 12 months (1,440 voyage records). It is mirrored in this repository at [`Datasets/ship_fuel_efficiency.csv`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/ship_fuel_efficiency.csv), so no API key or download step is required — same pattern as `Naval_Dataset.csv` in `NB02`.

We will train and evaluate **two real models** on this data:
- A **classification** model that predicts which fuel type (Diesel vs. HFO) a voyage used.
- A **regression** model that predicts fuel consumption from voyage conditions.

### Learning objectives

By the end of this class, students will be able to:
- Explain supervised, unsupervised, and reinforcement learning, and why data is split into training/validation/test sets.
- Define overfitting, underfitting, and the bias-variance tradeoff.
- Compute and interpret classification metrics (accuracy, precision, recall, F1, confusion matrix) and regression metrics (MAE, MSE, RMSE, R²).
- Recognize and avoid a data leakage pitfall using a real example.
- Load a real dataset, engineer features (one-hot encoding, scaling), and train/evaluate a classification model and a regression model.
- Use k-fold cross-validation to get a more robust performance estimate than a single train/test split.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap of NB01–NB06, today's roadmap | 5 min | Theory |
| 2 | Types of Machine Learning; key concepts (train/val/test, overfitting, bias-variance) | 15 min | Theory |
| 3 | Evaluation metrics: classification (worked example) and regression (formulas) | 15 min | Theory |
| 4 | Loading and exploring the real dataset (`ship_fuel_efficiency.csv`) | 15 min | Practice |
| 5 | Feature engineering & preprocessing, avoiding data leakage | 15 min | Theory + Practice |
| 6 | Classification in practice: predicting fuel type | 15 min | Practice |
| 7 | Regression in practice: predicting fuel consumption (Linear Regression vs. Random Forest) | 20 min | Practice |
| 8 | Cross-validation on the real model | 10 min | Theory + Practice |
| 9 | Practical applications of ML in naval/ocean engineering (overview) | 5 min | Theory |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.

---

## 1. Recap: where we are

- **`NB01`**: history of AI, weak vs. strong AI, why AI matters for naval/ocean engineering.
- **`NB02`**: Python and Colab essentials, NumPy arrays, and our first look at a naval dataset (`Naval_Dataset.csv`) with Pandas — `head()`, `describe()`, filtering, `groupby()`, `.corr()`, and simple plots.
- **`NB03`–`NB06`**: a deeper tooling pass before Machine Learning starts — NumPy array mechanics and engineering computation, a systematic Matplotlib class, and SciPy for naval engineering.
- **`NB07`** (today): the actual Machine Learning workflow — from "what is a model" to training and evaluating two real models on a real dataset.

We are inside **Block 2 — AI: Machine Learning** of the course roadmap (see `NB01` for the full four-block plan).

---

## 2. What is Machine Learning?

[Machine Learning (ML)](https://en.wikipedia.org/wiki/Machine_learning) is a subset of [Artificial Intelligence](https://en.wikipedia.org/wiki/Artificial_intelligence) that enables computers to **learn and make decisions from data**, instead of being explicitly programmed for every rule. It focuses on algorithms that `identify patterns in data and use those patterns to make predictions or decisions on new, unseen data`.

A few properties worth keeping in mind before we train anything:
- **Data-driven**: model quality is bounded by data quality — "garbage in, garbage out".
- **Adaptive**: models can be retrained as new data arrives; performance can degrade over time as conditions change (*model drift*).
- **Generalization is the goal**: a model that only memorizes its training examples is not useful — it must perform well on data it has never seen.

### Types of Machine Learning

| Type | Labeled data? | Goal | Common algorithms | Maritime example |
|---|:---:|---|---|---|
| **Supervised learning** | Yes | Learn a function that maps inputs → outputs | Linear/Logistic Regression, Decision Trees, Random Forest, SVM | Predict fuel consumption from voyage conditions |
| **Unsupervised learning** | No | Discover hidden structure/groupings | K-Means, PCA, Hierarchical Clustering | Segment ships into operational profiles |
| **Reinforcement learning** | Not exactly (reward signal) | Learn a policy that maximizes cumulative reward | Q-Learning, Policy Gradients, DQN | Autonomous route/speed optimization |

Today's class is entirely **supervised learning**: both problems we will solve (predicting fuel type, predicting fuel consumption) have a known, labeled target in the dataset.

> **Further reading**: [Supervised learning (Wikipedia)](https://en.wikipedia.org/wiki/Supervised_learning) · [Unsupervised learning (Wikipedia)](https://en.wikipedia.org/wiki/Unsupervised_learning) · [Reinforcement learning (Wikipedia)](https://en.wikipedia.org/wiki/Reinforcement_learning).

### Training, validation, and test sets

- The **training set** is used to fit the model — the data the algorithm actually learns from.
- The **validation set** is used to tune choices *about* the model (which algorithm, which hyperparameters) without touching the final evaluation data.
- The **test set** is used **once**, at the end, to estimate how the model will perform on genuinely new data. If you peek at the test set while tuning, your evaluation becomes optimistic and unreliable — this is a form of **data leakage**.

A common split ratio is 70/15/15 or 60/20/20. When a separate validation set is not practical (e.g., limited data), **k-fold cross-validation** is used instead — we will use it later today.

### Overfitting, underfitting, and the bias-variance tradeoff

- **Overfitting**: the model is too complex and learns the training data's noise, not just its pattern — great training performance, poor test performance. *High variance, low bias.*
- **Underfitting**: the model is too simple to capture the real pattern — poor performance on both training and test data. *High bias, low variance.*
- The **bias-variance tradeoff** is the balance between these two failure modes: as model complexity increases, bias tends to fall but variance tends to rise. The goal is the complexity level that minimizes total error on unseen data — usually found via cross-validation.

| Model complexity | Bias | Variance | Example |
|---|:---:|:---:|---|
| Too simple | High | Low | Linear regression on a strongly non-linear relationship |
| Too complex | Low | High | A very deep decision tree with no depth limit |
| Well balanced | Moderate | Moderate | Random Forest with tuned depth, or regularized linear model |

We will see this directly today: Linear Regression (simpler, higher bias) vs. Random Forest (more flexible, higher variance risk) on the same real prediction task.

> **Further reading**: [Overfitting (Wikipedia)](https://en.wikipedia.org/wiki/Overfitting) · [Bias–variance tradeoff (Wikipedia)](https://en.wikipedia.org/wiki/Bias%E2%80%93variance_tradeoff) · [Cross-validation (Wikipedia)](https://en.wikipedia.org/wiki/Cross-validation_%28statistics%29).

---

## 3. Evaluation metrics

### Classification metrics

Used when the target is categorical (e.g., Diesel vs. HFO, spam vs. not spam).

- **Accuracy**: $\dfrac{TP + TN}{TP + TN + FP + FN}$ — overall correctness; misleading on imbalanced data.
- **Precision**: $\dfrac{TP}{TP + FP}$ — of everything predicted positive, how much was actually positive.
- **Recall**: $\dfrac{TP}{TP + FN}$ — of everything actually positive, how much did we catch.
- **F1-score**: $2 \cdot \dfrac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — harmonic mean of precision and recall, useful under class imbalance.

#### Worked example: oil-spill detection (illustrative, hand-computed)

Suppose a satellite-imagery classifier is evaluated on 300 images (an imbalanced problem — real oil spills are rare) and produces this confusion matrix:

| | Predicted: spill | Predicted: no spill |
|---|:---:|:---:|
| **Actual: spill** | TP = 42 | FN = 14 |
| **Actual: no spill** | FP = 5 | TN = 239 |

$$
\text{Accuracy} = \frac{42 + 239}{300} \approx 93.7\% \qquad
\text{Precision} = \frac{42}{47} \approx 0.894 \qquad
\text{Recall} = \frac{42}{56} = 0.75 \qquad
F1 = 2 \cdot \frac{0.894 \cdot 0.75}{0.894 + 0.75} \approx 0.815
$$

`Even with 93.7% accuracy, recall is only 75%: one in four real spills is missed`. This is exactly why **accuracy alone is a poor metric for imbalanced, safety-critical problems** — a lesson we will apply for real later today. In practice we compute this with `confusion_matrix()` and `classification_report()`, not by hand — you will see both in a moment.

> **Further reading**: [Confusion matrix (Wikipedia)](https://en.wikipedia.org/wiki/Confusion_matrix) · [`sklearn.metrics.confusion_matrix` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) · [`sklearn.metrics.classification_report` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html).

### Regression metrics

Used when the target is continuous (e.g., fuel consumption, temperature).

| Metric | Formula | Reads as |
|---|---|---|
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | Average error size, same units as the target, robust to outliers |
| **MSE** (Mean Squared Error) | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ | Penalizes large errors more heavily |
| **RMSE** (Root Mean Squared Error) | $\sqrt{\text{MSE}}$ | Same units as the target, more sensitive to outliers than MAE |
| **R²** (Coefficient of Determination) | $1 - \dfrac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ | Proportion of variance explained; 1.0 = perfect, 0.0 = no better than predicting the mean |

> **Rule of thumb**: always look at MAE/RMSE *relative to the scale of the target*. An RMSE of 500 L is excellent if typical fuel consumption is 20,000 L, and terrible if it is 600 L. We will keep this in mind when we evaluate our own model later — the dataset's `fuel_consumption` column averages **~4,844 L** per voyage, with values ranging from about 238 to 24,650 L.

> **Further reading**: [Coefficient of determination (Wikipedia)](https://en.wikipedia.org/wiki/Coefficient_of_determination) · [`sklearn.metrics.r2_score` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) · [`sklearn.metrics.mean_absolute_error` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html).

---

## 4. Loading and exploring the real dataset

We now switch entirely to real data: **[Ship Fuel Consumption and CO2 Emissions Analysis](https://www.kaggle.com/datasets/jeleeladekunlefijabi/ship-fuel-consumption-and-co2-emissions-analysis)**, published on Kaggle, covering 120 vessels on Nigerian waterway routes over 12 months. It is already mirrored in this repository, so we load it the same way we loaded `Naval_Dataset.csv` in `NB02` — no Kaggle API key needed for this class.

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
print(fuel.shape)
fuel.head()

Each row is one voyage by one ship in one month. Columns:

| Column | Meaning |
|---|---|
| `ship_id` | Vessel identifier (120 unique ships) |
| `ship_type` | Oil Service Boat, Tanker Ship, Surfer Boat, or Fishing Trawler |
| `route_id` | One of 4 Nigerian waterway routes |
| `month` | Month of the voyage (Jan–Dec) |
| `distance` | Distance traveled (nautical miles) |
| `fuel_type` | Diesel or HFO (Heavy Fuel Oil) |
| `fuel_consumption` | Fuel burned on the voyage (liters) — **our regression target** |
| `CO2_emissions` | CO₂ emitted on the voyage (kg) |
| `weather_conditions` | Calm, Moderate, or Stormy |
| `engine_efficiency` | Engine efficiency (%) |

In [ ]:
fuel.info()

Now the numeric ranges and summary statistics:

In [ ]:
fuel.describe()

And the categories each column takes:

In [ ]:
for col in ["ship_type", "route_id", "fuel_type", "weather_conditions"]:
    print(fuel[col].value_counts())
    print()

### Quick exploratory questions

Before modeling anything, always look for patterns and sanity-check the data with `groupby()` — exactly like we practiced in `NB02`.

In [ ]:
fuel.groupby("ship_type")["fuel_consumption"].mean().sort_values(ascending=False)

Same question, grouped by weather instead:

In [ ]:
fuel.groupby("weather_conditions")["fuel_consumption"].mean().sort_values(ascending=False)

Visualize that same comparison as a boxplot:

In [ ]:
import matplotlib.pyplot as plt

fuel.boxplot(column="fuel_consumption", by="weather_conditions", figsize=(6, 4))
plt.title("Fuel consumption by weather condition")
plt.suptitle("")
plt.xlabel("Weather condition")
plt.ylabel("Fuel consumption (L)")
plt.show()

### A data leakage trap, found for real

Let's check how the numeric columns correlate with each other.

In [ ]:
numeric_cols = ["distance", "fuel_consumption", "CO2_emissions", "engine_efficiency"]
fuel[numeric_cols].corr()

Plot that correlation matrix so the leakage jumps out visually:

In [ ]:
plt.figure(figsize=(5.5, 5))
plt.imshow(fuel[numeric_cols].corr(), cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

You should see a correlation of about **0.997** between `fuel_consumption` and `CO2_emissions`. `That makes physical sense — CO₂ emitted is essentially a direct, near-linear function of fuel burned`. It also means: **if we used `CO2_emissions` as an input feature to predict `fuel_consumption`, the model would not really be learning anything — it would just be reading off an almost-equivalent value.** This is **data leakage**: a feature that (directly or indirectly) contains the answer.

We will exclude `CO2_emissions` from the regression features below for exactly this reason.

> **Further reading**: [Leakage in machine learning (Wikipedia)](https://en.wikipedia.org/wiki/Leakage_%28machine_learning%29).

---

## 5. Feature engineering and preprocessing

Real data is rarely ready for a model as-is. Two transformations we need today:

| Technique | Purpose | Here, applied to |
|---|---|---|
| **Encoding** | ML models need numbers, not text categories | `ship_type`, `route_id`, `fuel_type`, `weather_conditions` → one-hot columns via `pd.get_dummies()` |
| **Scaling** | Puts numeric features on comparable ranges (helps distance-based/linear models) | `distance`, `engine_efficiency` → `StandardScaler` |
| **Leakage removal** | Prevents a feature from encoding the answer | Drop `CO2_emissions` from the regression inputs (see above) |

`fuel.info()` above showed **no missing values** in this dataset, so we can skip imputation today — but in a real sensor-driven pipeline (e.g., onboard telemetry), missing/duplicate handling (`dropna()`, `fillna()`, `drop_duplicates()`) would normally come first.

---

## 6. Classification in practice: predicting fuel type

**Task**: given a voyage's ship type, route, distance, weather, and engine efficiency, predict whether it used **Diesel** or **HFO**. This is a genuine binary classification problem — fuel type is *not* fully determined by ship type alone (only `Surfer Boat` in this fleet uses exclusively Diesel; the other three ship types mix both fuels), so there is a real pattern for the model to learn, but it is not trivial.

> **Further reading**: [`sklearn.linear_model.LogisticRegression` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

clf_features = ["ship_type", "route_id", "distance", "weather_conditions", "engine_efficiency"]

X_clf = pd.get_dummies(fuel[clf_features], columns=["ship_type", "route_id", "weather_conditions"], drop_first=True)
y_clf = (fuel["fuel_type"] == "HFO").astype(int)  # 1 = HFO, 0 = Diesel

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler_clf = StandardScaler()
X_clf_train_scaled = scaler_clf.fit_transform(X_clf_train)
X_clf_test_scaled = scaler_clf.transform(X_clf_test)

print("Features used:", list(X_clf.columns))
print("Train size:", X_clf_train.shape, " Test size:", X_clf_test.shape)

> **Note**: `stratify=y_clf` keeps the Diesel/HFO proportion the same in both the train and test splits — important whenever classes are not perfectly balanced (here it's roughly 62%/38%).

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_clf_train_scaled, y_clf_train)
y_clf_pred = clf.predict(X_clf_test_scaled)

Now evaluate the classifier: confusion matrix and full metrics report:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(y_clf_test, y_clf_pred)
ConfusionMatrixDisplay(cm, display_labels=["Diesel", "HFO"]).plot(cmap="Blues")
plt.title("Fuel type classification — confusion matrix")
plt.show()

print(classification_report(y_clf_test, y_clf_pred, target_names=["Diesel", "HFO"]))

**Interpret your own output** (this will vary slightly run to run, since `LogisticRegression` and the split both have some randomness controlled by `random_state`, but should be broadly stable):
- Is accuracy meaningfully above the "always predict the majority class" baseline (~62%)?
- Are precision and recall for the minority class (whichever it is in your run) noticeably lower than for the majority class? Why might that be, given what we discussed about imbalance?
- Which route/ship-type features would you inspect next if recall for HFO were low?

---

## 7. Regression in practice: predicting fuel consumption

**Task**: predict `fuel_consumption` (liters) from voyage conditions — excluding `CO2_emissions` for the leakage reason established above, and excluding `ship_id` (just an identifier, not a real feature). `We'll train two models and compare them, exactly the bias-variance tradeoff discussed in Part 2`: a simple **Linear Regression** and a more flexible **Random Forest**.

> **Further reading**: [`sklearn.linear_model.LinearRegression` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [`sklearn.ensemble.RandomForestRegressor` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html).

In [ ]:
reg_features = ["ship_type", "distance", "fuel_type", "weather_conditions", "engine_efficiency"]

X_reg = pd.get_dummies(fuel[reg_features], columns=["ship_type", "fuel_type", "weather_conditions"], drop_first=True)
y_reg = fuel["fuel_consumption"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_reg_train_scaled = scaler_reg.fit_transform(X_reg_train)
X_reg_test_scaled = scaler_reg.transform(X_reg_test)

print("Features used:", list(X_reg.columns))

Train and compare both models on the same real prediction task:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_reg_train_scaled, y_reg_train)
    y_reg_pred = model.predict(X_reg_test_scaled)
    results.append({
        "Model": name,
        "MAE (L)": mean_absolute_error(y_reg_test, y_reg_pred),
        "RMSE (L)": mean_squared_error(y_reg_test, y_reg_pred) ** 0.5,
        "R2": r2_score(y_reg_test, y_reg_pred),
    })

results_df = pd.DataFrame(results)
results_df

Recall the dataset's `fuel_consumption` ranges roughly from 238 to 24,650 L, averaging ~4,844 L. Use that scale to judge whether the MAE/RMSE above are actually good — a metric only means something relative to the target's own range.

In [ ]:
best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X_reg_train_scaled, y_reg_train)
y_reg_pred = best_model.predict(X_reg_test_scaled)

plt.figure(figsize=(6, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.5)
lims = [y_reg.min(), y_reg.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual fuel consumption (L)")
plt.ylabel("Predicted fuel consumption (L)")
plt.title("Random Forest — predicted vs. actual")
plt.legend()
plt.show()

Points close to the red diagonal are voyages the model predicted well; points far from it are the model's biggest misses. If you have time, try `fuel.loc[y_reg_test.index]` filtered to the largest-error rows — is there a pattern (a specific ship type, route, or weather condition) among the worst predictions?

---

## 8. Cross-validation on the real model

`A single train/test split can be lucky or unlucky, especially with a dataset of this size`. **K-fold cross-validation** splits the data into *k* parts, trains on *k−1* of them, and tests on the remaining one — repeating *k* times so every row is used for testing exactly once — then reports the mean and spread of the score across folds.

| Technique | When to use | Notes |
|---|---|---|
| **K-Fold (k=5 or 10)** | General-purpose, most regression/classification tasks | Good bias/variance balance |
| **Stratified K-Fold** | Classification with class imbalance | Preserves class proportions in every fold — this is what we'd use for the fuel-type classifier |
| **Leave-One-Out (LOOCV)** | Very small datasets | Low bias, but slow and high variance; k = n |

> **Further reading**: [`sklearn.model_selection.cross_val_score` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) · [`sklearn.model_selection.KFold` documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html).

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42),
    X_reg, y_reg, cv=kf, scoring="r2"
)

print("R2 per fold:", cv_scores.round(3))
print(f"Mean R2: {cv_scores.mean():.3f}  (+/- {cv_scores.std():.3f})")

If the mean cross-validated R² is close to the single-split R² from Part 7, that's a good sign — the single split wasn't a lucky/unlucky draw. If the fold scores vary a lot, the model's performance is less stable than a single split suggested, which matters if this model were ever used operationally.

---

## 9. Machine Learning applications in naval and ocean engineering (overview)

Today's hands-on work covered *tabular* regression and classification. `ML also applies to several other data types relevant to this course` — we will return to computer vision and time-series/sequence problems in the upcoming **Deep Learning** block.

| Domain | Maritime/ocean example | Typical approach |
|---|---|---|
| Computer vision | Ship/obstacle detection from satellite or drone imagery; underwater ROV inspection | CNNs (Deep Learning block) |
| Time series / sequences | Wave height or sea-state forecasting; predictive maintenance from vibration sensors | RNNs, temporal models (Deep Learning block) |
| Clustering (unsupervised) | Segmenting ships or voyages into operational profiles without labels | K-Means, PCA |
| Natural language processing | Classifying maintenance logs, incident reports | Text classification |
| Anomaly detection | Flagging abnormal engine readings before failure | Isolation Forest, autoencoders |

---

## Class summary

- ML splits into supervised, unsupervised, and reinforcement learning; today was entirely supervised.
- Training/validation/test separation — and avoiding data leakage — is what makes an evaluation trustworthy. We found a real leakage trap (`CO2_emissions`) in today's own dataset.
- Classification metrics (accuracy, precision, recall, F1) and regression metrics (MAE, RMSE, R²) each answer a different question; always read them relative to the problem (class imbalance, target scale).
- We trained and evaluated two real models — a Logistic Regression classifier and a Linear Regression/Random Forest regression pair — on a real, published maritime dataset, not simulated data.
- K-fold cross-validation gives a more robust performance estimate than a single train/test split.

## For the next class (NB08)

We will go deeper into specific supervised algorithms (decision trees, ensembles, support vector machines) and start looking at unsupervised learning (clustering), filling a gap identified against the course's guía docente.

## Homework / Practice Ideas

1. Add `route_id` and/or `month` as extra encoded features to the regression model from Part 7 — does R² improve? Is the improvement worth the added complexity?
2. Repeat the classification task from Part 6, but predict `weather_conditions` (3 classes: Calm/Moderate/Stormy) instead of fuel type. Which metrics from Part 3 still apply directly, and which need adjusting for more than two classes?
3. In your own words, explain why excluding `CO2_emissions` from the regression features was necessary — and give one other example (naval or otherwise) of a feature that could leak the answer.
4. Compare the Linear Regression and Random Forest errors from Part 7 per ship type (hint: `groupby` the test-set errors by `ship_type`) — is one model much better for a particular ship type? Propose a reason why.
5. Change `cv=kf` in Part 8 to `cv=10` — how do the mean and standard deviation of R² change, and why would you expect that?

> ***As always: keep every question framed around what a naval engineer or fleet operator would actually want to know from this model.***

> For a deeper, textbook-level treatment of regression, classification, and cross-validation in one place, see the free textbook [*An Introduction to Statistical Learning*](https://www.statlearning.com/) (James, Witten, Hastie & Tibshirani).
